## STEP 1 GraphRAG

In [10]:
import json
import numpy as np
import chromadb
from openai import OpenAI
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123"))

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection(name="news")

print("초기화 완료!")

초기화 완료!


### Local Search 구현

In [11]:
def extract_entities(question):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": """질문에서 검색에 사용할 핵심 키워드를 추출하세요.
  인물, 조직, 국가, 사건, 주제 등을 쉼표로 구분해서 출력만 하세요.
  키워드가 없으면 질문의 핵심 명사를 추출하세요.
  절대 설명하지 말고 키워드만 출력하세요."""},
            {"role": "user", "content": question}
        ]
    )
    return [e.strip() for e in response.choices[0].message.content.split(",") if e.strip()]

def graph_local_search(question, hops=2):
    """엔티티 중심 그래프 탐색"""
    entities = extract_entities(question)
    print(f"추출된 엔티티: {entities}")

    graph_context = []

    with driver.session() as session:
        for entity in entities:
            # 1홉: 직접 연결
            result = session.run("""
                  MATCH (n {name: $name})-[r]-(m)
                  RETURN n.name AS source, type(r) AS relation, m.name AS target
              """, name=entity)

            for record in result:
                triple = f"{record['source']} -{record['relation']}-> {record['target']}"
                graph_context.append(triple)

            # 2홉: 간접 연결
            if hops >= 2:
                result = session.run("""
                      MATCH (n {name: $name})-[r1]-(mid)-[r2]-(end)
                      WHERE end.name <> n.name
                      RETURN n.name AS source, type(r1) AS r1, mid.name AS mid, type(r2) AS r2, end.name AS target
                      LIMIT 10
                  """, name=entity)

                for record in result:
                    triple = f"{record['source']} -{record['r1']}-> {record['mid']} -{record['r2']}->{record['target']}"
    graph_context.append(triple)

    # 중복 제거
    graph_context = list(set(graph_context))
    return graph_context


# 테스트
context = graph_local_search("이란 전쟁이 경제에 미치는 영향은?")
print(f"\n그래프 컨텍스트 {len(context)}개:")
for c in context:
    print(f"  {c}")


추출된 엔티티: ['이란', '전쟁', '경제', '영향']

그래프 컨텍스트 13개:
  이란 -드론_공격-> 아랍에미리트
  이란 -부인-> 핵무기 개발
  이란 -최고지도자-> 모즈타바 하메네이
  이란 -타격-> 이스라엘
  이란 -조건으로_제시하다-> 미국과 이스라엘의 공격 보장
  이란 -경고-> 미국·이스라엘 연계 은행
  이란 -무장_정파-> 헤즈볼라
  이란 -합동_공격-> 이스라엘
  이란 -공격-> 이스라엘
  이란 -공습-> 미국
  이란 -공동_공격-> 미국과 이스라엘
  이란 -소속-> 미나브 여자 초등학교
  이란 -무장_정파-> 헤즈볼라 -공습->이스라엘


### 벡터 검색 + 그래프 검색 결합

In [12]:
def graph_local_search(question, hops=2):
    entities = extract_entities(question)
    print(f"추출된 엔티티: {entities}")
    graph_context = []

    with driver.session() as session:
        for entity in entities:
            # 1홉 - CONTAINS로 부분 매칭
            result = session.run("""
                  MATCH (n)-[r]-(m)
                  WHERE n.name CONTAINS $name OR n.name = $name
                  RETURN n.name AS source, type(r) AS relation, m.name AS target
                  LIMIT 15
              """, name=entity)
            for record in result:
                triple = f"{record['source']} -{record['relation']}-> {record['target']}"
                graph_context.append(triple)

            # 2홉
            if hops >= 2:
                result = session.run("""
                      MATCH (n)-[r1]-(mid)-[r2]-(end)
                      WHERE (n.name CONTAINS $name OR n.name = $name) AND end.name <> n.name
                      RETURN n.name AS source, type(r1) AS r1, mid.name AS mid, type(r2) AS r2, end.name AS target
                      LIMIT 10
                  """, name=entity)
                for record in result:
                    triple = f"{record['source']} -{record['r1']}-> {record['mid']} -{record['r2']}->{record['target']}"
                    graph_context.append(triple)

    return list(set(graph_context))


answer = graph_local_search("이란 전쟁이 경제에 미치는 영향은?")
print(f"\n=== 답변 ===\n{answer}")

추출된 엔티티: ['이란', '전쟁', '경제', '영향']

=== 답변 ===
['이란 -타격-> 이스라엘 -시설_공격->레바논', '민생과 경제 산업 전반에 우려 -지적하다-> 이재명 대통령 -주재->수석보좌관회의', '이란 -타격-> 이스라엘 -우려하다->자신들을 다시 공격할 가능성', '경제더하기연구소 -대표-> 이용우', '이란 -무장_정파-> 헤즈볼라', '이란의 핵 능력 발전 노력 -사용-> 탈레간 핵시설', '이란 -타격-> 이스라엘 -작전_확대->레바논', '이란 -타격-> 이스라엘 -주장->AMAD 프로젝트', '이란 -공동_공격-> 미국과 이스라엘', '민생과 경제 산업 전반에 우려 -지적하다-> 이재명 대통령 -강조하다->추경의 필요성', '이란 -타격-> 이스라엘 -공습->모즈타바 하메네이', '민생과 경제 산업 전반에 우려 -지적하다-> 이재명 대통령 -주재하다->수석 보좌관 회의', '이란 -드론_공격-> 아랍에미리트', '이란 -타격-> 이스라엘 -공격->헤즈볼라', '이란 -최고지도자-> 모즈타바 하메네이', '이란의 정당한 권리 -강조하다-> 마수드 페제시키안', '이란 -공습-> 미국', '이란 -부인-> 핵무기 개발', '이란 -조건으로_제시하다-> 미국과 이스라엘의 공격 보장', '이란 -타격-> 이스라엘', '이란 -타격-> 이스라엘 -합동_작전->헤즈볼라', '우리 경제 -영향_미침-> 중동지역', '민생과 경제 산업 전반에 우려 -지적하다-> 이재명 대통령 -논의->자본시장의 안정화 방안', '이란 -소속-> 미나브 여자 초등학교', '이란 -타격-> 이스라엘 -공습->헤즈볼라', '민생과 경제 산업 전반에 우려 -지적하다-> 이재명 대통령', '이란 -경고-> 미국·이스라엘 연계 은행', '이란 -공격-> 이스라엘', '이란 -합동_공격-> 이스라엘', '경제관계장관회의 -발언-> 김영훈', '민생과 경제 산업 전반에 우려 -지적하다-> 이재명 대통령 -발언->국민의 물가 부담 완화와 민생 안정', '민생과 경제 

### Naive RAG vs GraphRAG 비교

In [13]:
def naive_rag(question):
    q_resp = client.embeddings.create(input=[question], model="text-embedding-3-small")
    results = collection.query(query_embeddings=[q_resp.data[0].embedding], n_results=3)
    context = "\n\n".join(
        [f"[참고자료 {i + 1}]\n{doc}" for i, doc in enumerate(results['documents'][0])])
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "참고자료를 기반으로 답변하세요. 없는 내용은 '해당 정보를 찾을 수  없습니다'라고 하세요."},
            {"role": "user", "content": f"{context}\n\n질문: {question}"}
        ]
    )
    return response.choices[0].message.content


questions = [
    "이란 전쟁이 한국 경제에 미치는 영향은?",
    "트럼프와 이란의 관계를 설명해줘",
    "중동 사태와 관련된 주요 인물들은?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"\n[Naive RAG]")
    print(naive_rag(q)[:200])
    print(f"\n[GraphRAG]")
    print(graph_local_search(q)[:200])
    print("=" * 60)



Q: 이란 전쟁이 한국 경제에 미치는 영향은?

[Naive RAG]
해당 정보를 찾을 수 없습니다.

[GraphRAG]
추출된 엔티티: ['이란 전쟁', '한국 경제', '영향']
[]

Q: 트럼프와 이란의 관계를 설명해줘

[Naive RAG]
도널드 트럼프 대통령과 이란의 관계는 긴장감으로 가득 차 있었습니다. 트럼프 행정부는 이란의 핵 및 미사일 프로그램을 강력히 비판하며 이를 폐기하려는 노력을 기울였습니다. 그는 이란의 지도자들이 핵무기를 가질 경우 발생할 수 있는 위험에 대해 경고하며, 이를 이유로 공습 작전의 정당성을 주장했습니다.

에 의하면, 트럼프 행정부는 이란과의 직접적인 대화는 

[GraphRAG]
추출된 엔티티: ['트럼프', '이란', '관계']
['도널드 트럼프 -관여-> 미국 행정부', '우리은행 관계자 -확립-> 소비자 중심 금융 문화', '이란 -타격-> 이스라엘 -시설_공격->레바논', '이란 -타격-> 이스라엘 -우려하다->자신들을 다시 공격할 가능성', '이란 -무장_정파-> 헤즈볼라', '이란의 핵 능력 발전 노력 -사용-> 탈레간 핵시설', '트럼프 행정부 -속도전-> 관세정책', '이란 -타격-> 이스라엘 -작전_확대->레바논', '이란 -공동_공격-> 미국과 이스라엘', '이란 -타격-> 이스라엘 -주장->AMAD 프로젝트', '이란 -타격-> 이스라엘 -공습->모즈타바 하메네이', '이란 -드론_공격-> 아랍에미리트', '이란 -타격-> 이스라엘 -공격->헤즈볼라', '이란 -최고지도자-> 모즈타바 하메네이', '이란의 정당한 권리 -강조하다-> 마수드 페제시키안', '이란 -공습-> 미국', '이란 -부인-> 핵무기 개발', '이란 -조건으로_제시하다-> 미국과 이스라엘의 공격 보장', '이란 -타격-> 이스라엘', '이란 -타격-> 이스라엘 -합동_작전->헤즈볼라', '이란 -소속-> 미나브 여자 초등학교', '이란 -타격-> 이스라엘 -공습->헤즈볼라', '이란 -경고-> 미국·이스라엘 연계 은행', 

## STEP 2 Global Search (커뮤니티 기반 검색)

In [14]:
import json
import chromadb
from openai import OpenAI
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123"))
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection(name="news")

### 커뮤니티 탐지

In [15]:
def detect_communities():
    """연결된 노드 그룹을 커뮤니티로 추출"""
    with driver.session() as session:
        # 모든 노드와 연결 가져오기
        result = session.run("""
              MATCH (n)-[r]-(m)
              RETURN n.name AS source, type(r) AS relation, m.name AS target
          """)

        edges = []
        for record in result:
            edges.append((record['source'], record['relation'], record['target']))

    # 간단한 Union-Find로 커뮤니티 탐지
    parent = {}

    def find(x):
        if x not in parent:
            parent[x] = x
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(x, y):
        px, py = find(x), find(y)
        if px != py:
            parent[px] = py

    for src, rel, tgt in edges:
        union(src, tgt)

    # 커뮤니티별 노드와 관계 정리
    communities = {}
    for src, rel, tgt in edges:
        root = find(src)
        if root not in communities:
            communities[root] = {"nodes": set(), "relations": []}
        communities[root]["nodes"].add(src)
        communities[root]["nodes"].add(tgt)
        communities[root]["relations"].append(f"{src} -{rel}-> {tgt}")

    # 크기순 정렬
    sorted_communities = sorted(communities.values(), key=lambda x: len(x["nodes"]), reverse=True)
    return sorted_communities


communities = detect_communities()
print(f"총 커뮤니티 수: {len(communities)}\n")
for i, comm in enumerate(communities[:5]):
    print(f"--- 커뮤니티 {i + 1} ({len(comm['nodes'])}개 노드) ---")
    print(f"노드: {list(comm['nodes'])[:8]}")
    print(f"관계 수: {len(comm['relations'])}")
    print()

총 커뮤니티 수: 180

--- 커뮤니티 1 (61개 노드) ---
노드: ['법 왜곡죄', '이재명 대통령', '법 위반 사건', '공정사회포럼', '윤석열', '박영재', '전국법원장회의', '사회민주당']
관계 수: 122

--- 커뮤니티 2 (30개 노드) ---
노드: ['아랍에미리트', '핵무기 개발', '탈레간 복합 시설', '관세 부과', '이란', 'IAEA', '유세프 페제시키안', '미국·이스라엘 연계 은행']
관계 수: 76

--- 커뮤니티 3 (13개 노드) ---
노드: ['공정거래법', '2026 엔씨 경영전략 간담회', '모바일 캐주얼 사업', '아키에이지 워', '저작권 침해', '모바일 캐주얼', '리니지2M', '엑스엘게임즈']
관계 수: 24

--- 커뮤니티 4 (10개 노드) ---
노드: ['삼성생명', '폰(4a)', '낫싱', '유배당 보험', '국내 시장', '삼성화재', '애플', '31%']
관계 수: 22

--- 커뮤니티 5 (10개 노드) ---
노드: ['징역 1년', '남태현', '법원', '인명 피해 없음', '검찰', '서울경찰청 형사기동대', '남태현 측 법률대리인', '강변북로 동작대교']
관계 수: 18



### 각 커뮤니티 요약 생성

In [16]:
def summarize_communities(communities, max_communities=10):
    """각 커뮨니티를 LLM으로 요약"""
    summaries = []

    for i, comm in enumerate(communities[:max_communities]):
        if len(comm['nodes']) < 3:  # 너무 작은 커뮤니티 스킵
            continue

        relations_text = "\n".join(comm['relations'][:20])  # 최대 20개 관계

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system",
                 "content": "아래 엔티티 관계들을 분석해서 이 그룹이 어떤 주제/사건에 대한 것인지 2-3문장으로 요약하세요."},
                {"role": "user",
                 "content": f"엔티티: {list(comm['nodes'])[:15]}\n\n관계:\n{relations_text}"}
            ]
        )

        summary = response.choices[0].message.content
        summaries.append({
            "id": i,
            "nodes": list(comm['nodes']),
            "summary": summary,
            "size": len(comm['nodes'])
        })
        print(f"커뮤니티 {i + 1} ({len(comm['nodes'])}개 노드): {summary[:80]}...")

    return summaries


summaries = summarize_communities(communities)

커뮤니티 1 (61개 노드): 이 그룹은 이재명 대통령과 관련된 정치적 사건과 경제 정책 논의를 다루고 있습니다. 특히, 이재명 대통령이 주재한 수석보좌관 회의 및 간담회에서...
커뮤니티 2 (30개 노드): 이 엔티티 관계들은 주로 중동 지역의 긴장 상황과 관련된 내용으로 보입니다. 특히, 이란의 핵무기 개발과 이에 대한 미국과 이스라엘의 반응, 그...
커뮤니티 3 (13개 노드): 이 그룹은 엔씨소프트의 경영 전략과 관련된 내용으로, 특히 모바일 카주얼 사업의 성장을 통한 2030년 매출 5조원 목표 설정과 관련된 전략 발...
커뮤니티 4 (10개 노드): 이 그룹은 삼성생명, 삼성전자 및 낫싱과 관련된 금융 및 기술 시장의 상호작용을 다루고 있습니다. 삼성생명과 삼성전자는 서로의 지분을 보유하며 ...
커뮤니티 5 (10개 노드): 이 엔티티 관계는 남태현이라는 인물이 음주 운전과 마약류 관리법 위반으로 인해 법적 문제에 휘말린 사건을 다루고 있습니다. 그 결과로 남태현은 ...
커뮤니티 6 (8개 노드): 이 그룹은 서서울미술관의 개관과 관련된 정보들로 구성되어 있습니다. 박나운이 관장으로 활동하며, 서울시립미술관의 분관으로 운영되는 서서울미술관은...
커뮤니티 7 (7개 노드): 이 그룹은 백태웅이라는 인물의 경력과 관련된 사건들을 다루고 있습니다. 백태웅은 하와이대 및 고려대 로스쿨의 교수로 재직하면서, 사노맹이라는 조...
커뮤니티 8 (7개 노드): 이 그룹은 정부가 경제와 사회 안정성을 확보하기 위해 다양한 정책과 계획을 추진하는 상황을 다루고 있습니다. 이러한 정책에는 석유 가격 조정, ...
커뮤니티 9 (7개 노드): 이 그룹은 할리우드 유명 인물 바브라 스트라이샌드의 경력과 그가 수상한 여러 영화상, 특히 명예 황금종려상, 오스카상, 골든글로브에 대한 내용을...
커뮤니티 10 (6개 노드): 이 엔티티와 관계들은 한미 연합작전 훈련 및 임무 수행에 대한 내용을 다루고 있습니다. 진영승과 제이비어 브런슨이 백령도를 방문하여 한미 장

### Global Search 구현

In [17]:
def graph_global_search(question):
    """커뮤니티 요약 기반 검색"""
    print(f"질문: {question}\n")

    # 모든 커뮤니티 요약을 하나로
    all_summaries = "\n\n".join([
        f"[주제 그룹 {s['id'] + 1}] ({s['size']}개 엔티티)\n{s['summary']}"
        for s in summaries
    ])

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": """아래는 뉴스 기사들에서 추출한 주제 그룹별 요약입니다.
  이 요약들을 종합해서 질문에 포괄적으로 답변하세요.
  관련 없는 그룹은 무시하세요. 없는 내용은 지어내지 마세요."""},
            {"role": "user", "content": f"{all_summaries}\n\n질문: {question}"}
        ]
    )

    return response.choices[0].message.content

### Local Search vs Global Search vs Naive RAG 비교

In [18]:
def naive_rag(question):
    q_resp = client.embeddings.create(input=[question], model="text-embedding-3-small")
    results = collection.query(query_embeddings=[q_resp.data[0].embedding], n_results=3)
    context = "\n\n".join(results['documents'][0])
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "참고자료를 기반으로 답변하세요. 없는 내용은 '해당 정보를 찾을 수 없습니다'라고 하세요."},
            {"role": "user", "content": f"{context}\n\n질문: {question}"}
        ]
    )
    return response.choices[0].message.content


questions = [
    "오늘 뉴스의 전체적인 흐름을 요약해줘",
    "현재 가장 중요한 이슈 3가지는?",
    "트럼프가 이란에 대해 뭐라 했어?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"\n[Naive RAG]")
    print(naive_rag(q)[:150])
    print(f"\n[Local Search]")
    print(graph_local_search(q)[:150])
    print(f"\n[Global Search]")
    print(graph_global_search(q)[:150])
    print("=" * 60)



Q: 오늘 뉴스의 전체적인 흐름을 요약해줘

[Naive RAG]
오늘 뉴스에서는 CBS노컷뉴스와 MBC뉴스가 각각 제보를 통해 사회의 다양한 이슈를 다루고 있으며, 두 매체 모두 제보를 기다리고 있다는 내용을 전달하고 있습니다. 또한, 한미 간의 협의에 대한 언급이 있어, 관련 계획은 없는 상황임을 알리고 계속해서 협의를 이어나갈 

[Local Search]
추출된 엔티티: ['뉴스', '흐름', '요약']
[]

[Global Search]
질문: 오늘 뉴스의 전체적인 흐름을 요약해줘

오늘 뉴스의 전체적인 흐름은 다음과 같습니다:

1. **정치 및 경제 상황**: 이재명 대통령이 주재한 회의에서 경제 상황과 자본시장의 안정화 방안에 대한 논의가 이루어졌습니다. 이 회의에는 금융위원장 및 금융감독원장 등 주요 인사들이 참석하여 민생 및 경제 산업 전

Q: 현재 가장 중요한 이슈 3가지는?

[Naive RAG]
1. AI와 데이터 보안 및 거버넌스 체계 구축: AI 시대에 중요성을 더하고 있는 보안 및 거버넌스 체계의 구축 방안에 대한 논의가 필요합니다.

2. 공급망 보안 대응: 민간 중심의 뉴스페이스 시대에서 위성 제작 및 운영 과정에서 발생할 수 있는 보안 위협에 대한 

[Local Search]
추출된 엔티티: ['이슈']
[]

[Global Search]
질문: 현재 가장 중요한 이슈 3가지는?

현재 가장 중요한 이슈 세 가지는 다음과 같습니다:

1. **이재명 대통령의 경제 정책 및 민생 안정**: 이재명 대통령은 최근 수석보좌관 회의에서 민생과 경제 산업 전반에 대한 우려를 표명하며 자본시장의 안정화 방안을 논의했습니다. 금융위원장 및 금융감독원장 등 주

Q: 트럼프가 이란에 대해 뭐라 했어?

[Naive RAG]
도널드 트럼프 미국 대통령은 이란에 대해 “미친 사람들이 핵무기를 가지면 나쁜 일이 벌어진다”며 이란의 핵·미사일 프로그램 폐기를 명분으로 한 공습 상황에 대해 “지금은 아주 좋은 상태”라고 밝혔습니다.

[Loca

## STEP 3 GraphRAG 통합 파이프라인

In [19]:
import json
import re
import numpy as np
import chromadb
from openai import OpenAI
from neo4j import GraphDatabase
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi

load_dotenv()
client = OpenAI()
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123"))
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection(name="news")

# BM25 준비
all_data = collection.get(include=["documents", "metadatas"])
all_docs = all_data['documents']
all_metas = all_data['metadatas']
tokenized_docs = [doc.split() for doc in all_docs]
bm25 = BM25Okapi(tokenized_docs)

print(f"초기화 완료! 청크: {len(all_docs)}개")

초기화 완료! 청크: 613개


#### 질문 유형 판단기

In [20]:
def classify_question(question):
    """질문이 구체적(local)인지 포괄적(global)인지 판단"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": """질문 유형을 판단하세요.
- "local": 특정 인물, 사건, 조직에 대한 구체적 질문 (예: "트럼프가 뭐라 했어?", "천궁 미사일 성공률은?")
- "global": 전체 요약, 흐름, 주요 이슈 등 포괄적 질문 (예: "오늘 뉴스 요약해줘", "가장 중요한 이슈는?")
local 또는 global 한 단어만 출력하세요."""},
            {"role": "user", "content": question}
        ]
    )
    answer = response.choices[0].message.content.strip().lower()
    return "global" if "global" in answer else "local"

#### 각 검색 함수

In [21]:
def extract_entities(question):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": """질문에서 검색에 사용할 핵심 키워드를 추출하세요.
    인물, 조직, 국가, 사건, 주제 등을 쉼표로 구분해서 출력만 하세요.
    절대 설명하지 말고 키워드만 출력하세요."""},
            {"role": "user", "content": question}
        ]
    )
    return [e.strip() for e in response.choices[0].message.content.split(",") if e.strip()]


def graph_local_search(question, hops=2):
    entities = extract_entities(question)
    graph_context = []
    with driver.session() as session:
        for entity in entities:
            result = session.run("""
              MATCH (n)-[r]-(m)
              WHERE n.name CONTAINS $name OR n.name = $name
              RETURN n.name AS source, type(r) AS relation, m.name AS target
              LIMIT 15
          """, name=entity)
            for record in result:
                graph_context.append(
                    f"{record['source']} -{record['relation']}-> {record['target']}")
            if hops >= 2:
                result = session.run("""
                  MATCH (n)-[r1]-(mid)-[r2]-(end)
                  WHERE (n.name CONTAINS $name OR n.name = $name) AND end.name <> n.name
                  RETURN n.name AS source, type(r1) AS r1, mid.name AS mid, type(r2) AS r2, end.name AS target
                  LIMIT 10
              """, name=entity)
                for record in result:
                    graph_context.append(
                        f"{record['source']} -{record['r1']}-> {record['mid']} -{record['r2']}{record['target']}")
    return list(set(graph_context))


def hybrid_search(question, top_k=5, alpha=0.5):
    tokenized_query = question.split()
    bm25_scores = bm25.get_scores(tokenized_query)
    if max(bm25_scores) > 0:
        bm25_scores = bm25_scores / max(bm25_scores)
    q_resp = client.embeddings.create(input=[question], model="text-embedding-3-small")
    vec_results = collection.query(
        query_embeddings=[q_resp.data[0].embedding],
        n_results=len(all_docs),
        include=["distances"]
    )
    vec_scores = np.zeros(len(all_docs))
    for i, doc_id in enumerate(vec_results['ids'][0]):
        idx = all_data['ids'].index(doc_id)
        vec_scores[idx] = 1 - vec_results['distances'][0][i]
    if max(vec_scores) > 0:
        vec_scores = vec_scores / max(vec_scores)
    combined = alpha * vec_scores + (1 - alpha) * bm25_scores
    top_indices = np.argsort(combined)[::-1][:top_k]
    return [all_docs[i] for i in top_indices]

#### 커뮤니티 요약 사전 생성

In [22]:
def detect_communities():
    with driver.session() as session:
        result = session.run("""
          MATCH (n)-[r]-(m)
          RETURN n.name AS source, type(r) AS relation, m.name AS target
      """)
        edges = [(r['source'], r['relation'], r['target']) for r in result]

    parent = {}

    def find(x):
        if x not in parent: parent[x] = x
        if parent[x] != x: parent[x] = find(parent[x])
        return parent[x]

    def union(x, y):
        px, py = find(x), find(y)
        if px != py: parent[px] = py

    for src, rel, tgt in edges:
        union(src, tgt)

    communities = {}
    for src, rel, tgt in edges:
        root = find(src)
        if root not in communities:
            communities[root] = {"nodes": set(), "relations": []}
        communities[root]["nodes"].update([src, tgt])
        communities[root]["relations"].append(f"{src} -{rel}-> {tgt}")

    return sorted(communities.values(), key=lambda x: len(x["nodes"]), reverse=True)


def summarize_communities(communities, max_communities=10):
    summaries = []
    for i, comm in enumerate(communities[:max_communities]):
        if len(comm['nodes']) < 3: continue
        relations_text = "\n".join(comm['relations'][:20])
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "엔티티 관계들을 분석해서 이 그룹의 주제를 2-3문장으로 요약하세요."},
                {"role": "user",
                 "content": f"엔티티: {list(comm['nodes'])[:15]}\n\n관계:\n{relations_text}"}
            ]
        )
        summaries.append(
            {"id": i, "summary": response.choices[0].message.content, "size": len(comm['nodes'])})
    return summaries


communities = detect_communities()
summaries = summarize_communities(communities)
print(f"커뮤니티 {len(summaries)}개 요약 완료!")

커뮤니티 10개 요약 완료!


#### 최종 통합 파이프라인

In [23]:
def graph_rag_pipeline(question):
    # 1. 질문 유형 판단
    q_type = classify_question(question)
    print(f"질문: {question}")
    print(f"유형: {q_type}\n")

    # 2. 유형에 따라 검색
    if q_type == "local":
        # Local: 그래프 탐색 + Hybrid 검색
        graph_context = graph_local_search(question)
        vector_docs = hybrid_search(question, top_k=3)

        context = f"""[그래프 관계 ({len(graph_context)}개)]
{chr(10).join(graph_context[:20])}

[관련 문서]
{chr(10).join(vector_docs[:3])}"""

    else:
        # Global: 커뮤니티 요약 + Hybrid 검색
        community_text = "\n\n".join([
            f"[주제 {s['id'] + 1}] {s['summary']}" for s in summaries
        ])
        vector_docs = hybrid_search(question, top_k=3)

        context = f"""[주제별 요약]
{community_text}

[관련 문서]
{chr(10).join(vector_docs[:3])}"""

    # 3. LLM 답변 생성
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "제공된 정보를 종합해서 질문에 답변하세요. 없는 내용은 지어내지마세요."},
            {"role": "user", "content": f"{context}\n\n질문: {question}"}
        ]
    )

    return response.choices[0].message.content


# 테스트
questions = [
    "트럼프가 이란에 대해 뭐라 했어?",
    "오늘 뉴스의 전체적인 흐름을 요약해줘",
    "이란 전쟁이 한국 경제에 미치는 영향은?",
    "현재 가장 중요한 이슈 3가지는?",
    "천궁 미사일의 요격 성공률은?",
]

for q in questions:
    answer = graph_rag_pipeline(q)
    print(f"답변: {answer[:200]}")
    print("=" * 60)


질문: 트럼프가 이란에 대해 뭐라 했어?
유형: local

답변: 도널드 트럼프 미국 대통령은 이란에 대한 공습 작전과 관련해 "미친 사람들이 핵무기를 가지면 나쁜 일이 벌어진다"며 이란의 핵·미사일 프로그램 폐기를 명분으로 공습 상황에 대해 "지금은 아주 좋은 상태"라고 밝혔습니다. 또 이란 지도자에게 "되려는 자는 결국 죽음 맞이하게 될 것"이라고 경고했습니다.
질문: 오늘 뉴스의 전체적인 흐름을 요약해줘
유형: global

답변: 오늘 뉴스의 전체적인 흐름은 여러 주제를 포함하여 경제, 정치, 법적 사건, 문화, 군사 협력 등 다양한 분야에서의 활동과 이슈가 강조되고 있습니다.

1. **경제 및 정치**: 이재명 대통령이 민생과 경제 문제를 언급하며 자본시장 안정화 방안을 논의하고 있습니다. 여러 경제 관련 인사와 간담회를 통해 의견을 교환하고 있습니다.

2. **군사 및 국제 
질문: 이란 전쟁이 한국 경제에 미치는 영향은?
유형: global

답변: 이란과 관련된 전쟁 상황이 한국 경제에 미치는 영향은 여러 가지가 있습니다. 첫째로, 중동 지역에서의 정치적 및 군사적 긴장은 유가 변동에 직접적인 영향을 미칠 수 있습니다. 예를 들어, 호르무즈 해협에서의 유조선 호송과 같은 이란의 군사적 동향이 미국 및 기타 국가들과의 긴장 관계와 맞물리면서 유가 상승을 초래할 경우, 한국의 에너지 수입 비용이 증가하게
질문: 현재 가장 중요한 이슈 3가지는?
유형: global

답변: 현재 가장 중요한 이슈 3가지는 다음과 같습니다.

1. **대한민국 정치 및 경제 상황**: 이재명 대통령의 민생과 경제에 대한 우려가 커지고 있으며, 경제 정책과 자본시장 안정화 방안에 대한 논의가 활발해지고 있습니다. 이와 관련하여 여러 경제 인사와 단체와의 간담회도 진행되고 있어 경제 회복에 대한 관심이 집중되고 있습니다.

2. **아랍에미리트의 
질문: 천궁 미사일의 요격 성공률은?
유형: local

답변: 제공된 정보에는 천궁 미사일의 요격 성공률에 대한 구체적

## Step 4: Naive RAG vs Advanced RAG vs GraphRAG 최종 비교

#### 3가지 파이프라인 정의

In [24]:
# 1. Naive RAG
def naive_rag(question):
    q_resp = client.embeddings.create(input=[question], model="text-embedding-3-small")
    results = collection.query(query_embeddings=[q_resp.data[0].embedding], n_results=3)
    context = "\n\n".join(results['documents'][0])
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "참고자료를 기반으로 답변하세요. 없는 내용은 '해당 정보를 찾을 수 없습니다'라고 하세요."},
            {"role": "user", "content": f"{context}\n\n질문: {question}"}
        ]
    )
    return response.choices[0].message.content


def generate_multi_queries(question, n=3):
  response = client.chat.completions.create(
      model="gpt-4o-mini",
      messages=[
          {"role": "system", "content": f"주어진 질문을 {n}가지 다른 표현으로 바꿔주세요. 각 줄에 하나씩만 출력하세요."},
          {"role": "user", "content": question}
      ]
  )
  queries = response.choices[0].message.content.strip().split("\n")
  return [q.strip().lstrip("0123456789.-) ") for q in queries if q.strip()]

# 2. Advanced RAG (Multi-Query + Hybrid + Rerank)
def advanced_rag(question):
    # Multi-Query
    queries = [question] + generate_multi_queries(question)

    # Hybrid Search per query
    all_results = []
    seen = set()
    for q in queries:
        results = hybrid_search(q, top_k=5)
        for doc in results:
            key = doc[:50]
            if key not in seen:
                seen.add(key)
                all_results.append(doc)

    # Rerank
    candidate_docs = all_results[:15]
    doc_list = "\n\n".join([f"[문서 {i + 1}] {doc}" for i, doc in enumerate(candidate_docs)])
    rerank_resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "질문에 가장 관련성 높은 문서 3개를 골라 번호만 출력. 쉼표로 구분. 예:  3, 7, 1"},
            {"role": "user", "content": f"질문: {question}\n\n{doc_list}"}
        ]
    )
    indices = [int(x.strip()) - 1 for x in rerank_resp.choices[0].message.content.strip().split(",")
               if
               x.strip().isdigit()]
    final_docs = [candidate_docs[i] for i in indices if i < len(candidate_docs)]

    context = "\n\n".join(final_docs[:3])
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "참고자료를 기반으로 답변하세요. 없는 내용은 '해당 정보를 찾을 수 없습니다'라고 하세요."},
             {"role": "user", "content": f"{context}\n\n질문: {question}"}
        ]
    )
    return response.choices[0].message.content

#### 5가지 질문으로 비교

In [25]:
questions = [
    # 구체적 사실 확인
    "천궁 미사일의 요격 성공률은?",
    # 특정 인물 + 관계
    "트럼프와 이란의 관계를 설명해줘",
    # 간접 추론
    "이란 전쟁이 한국 경제에 미치는 영향은?",
    # 전체 요약
    "오늘 뉴스의 전체적인 흐름을 요약해줘",
    # 다수 엔티티 나열
    "중동 사태와 관련된 주요 인물들은?",
]

for q in questions:
    print(f"\n{'=' * 60}")
    print(f"Q: {q}")
    print(f"{'=' * 60}")

    print("\n[1. Naive RAG]")
    print(naive_rag(q)[:200])

    print("\n[2. Advanced RAG]")
    print(advanced_rag(q)[:200])

    print("\n[3. GraphRAG]")
    print(graph_rag_pipeline(q)[:200])


Q: 천궁 미사일의 요격 성공률은?

[1. Naive RAG]
해당 정보를 찾을 수 없습니다.

[2. Advanced RAG]
해당 정보를 찾을 수 없습니다.

[3. GraphRAG]
질문: 천궁 미사일의 요격 성공률은?
유형: local

제공된 정보에는 천궁 미사일의 요격 성공률에 대한 직접적인 언급이 없습니다. 그러나 아랍에미리트가 이란의 공격을 요격한 성공률은 90% 이상으로 설명되어 있습니다. 천궁 미사일의 요격 성공률에 대한 정보는 포함되어 있지 않으므로, 해당 수치는 확인할 수 없습니다.

Q: 트럼프와 이란의 관계를 설명해줘

[1. Naive RAG]
도널드 트럼프 대통령과 이란의 관계는 긴장된 상황으로 특징지어집니다. 트럼프 행정부는 이란의 핵 프로그램과 미사일 개발을 강력히 반대하며, 이란에 대한 강도 높은 제재를 부과했습니다. 이란과의 공식적인 대화는 없었으며, 백악관은 이란과의 접촉이 없다고 밝혔다. 트럼프는 이란 지도자들에 대해 강한 반감을 표현하며, 이란이 핵무기를 가지는 것을 우려했습니다. 

[2. Advanced RAG]
도널드 트럼프 대통령과 이란의 관계는 주로 이란의 핵·미사일 프로그램과 관련된 긴장으로 특징지어집니다. 트럼프는 이란의 지도부가 핵무기를 보유하는 것을 엄격히 반대하며, 이란의 군사적 활동에 대해 강한 입장을 보이고 있습니다. 그는 이란 공습 작전과 관련하여 이란의 지도부가 무너지고 있으며 지도자가 되려는 자는 결국 죽음을 맞이할 것이라고 경고했습니다. 이

[3. GraphRAG]
질문: 트럼프와 이란의 관계를 설명해줘
유형: local

트럼프와 이란의 관계는 긴장과 갈등으로 특징지어집니다. 도널드 트럼프 대통령은 이란의 핵 프로그램과 미사일 개발에 강력한 반대 입장을 보였으며, 이란이 "미친 사람들이 핵무기를 가지면 나쁜 일이 벌어진다"고 언급하며 공습 작전의 필요성을 강조했습니다. 이러한 발언은 이란의 군사적 행보와 관련된 우려를 반영하며, 트럼프의 강경한 외교 정책을 뒷받침하고 있습니

Q: 이

#### 결과 정리표 자동 생성

In [26]:
# 각 방식의 답변을 LLM이 평가
def evaluate(question, answers):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": """세 가지 RAG 방식의 답변을 비교 평가하세요.
각 답변에 대해:
- 정확성 (1-5): 사실에 부합하는가
- 완성도 (1-5): 질문에 충분히 답했는가
- 추론력 (1-5): 간접적 관계까지 파악했는가
형식: 방식명 | 정확성 | 완성도 | 추론력 | 한줄평"""},
            {"role": "user", "content": f"""질문: {question}

[Naive RAG] {answers[0][:300]}

[Advanced RAG] {answers[1][:300]}

[GraphRAG] {answers[2][:300]}"""}
        ]
    )
    return response.choices[0].message.content


for q in questions:
    a1 = naive_rag(q)
    a2 = advanced_rag(q)
    a3 = graph_rag_pipeline(q)

    print(f"\nQ: {q}")
    print(evaluate(q, [a1, a2, a3]))
    print("-" * 60)


질문: 천궁 미사일의 요격 성공률은?
유형: local


Q: 천궁 미사일의 요격 성공률은?
Naive RAG | 정확성: 1 | 완성도: 1 | 추론력: 1 | 한줄평: 정보 부재로 인해 질문에 대한 답변이 전혀 제공되지 않았다.

Advanced RAG | 정확성: 1 | 완성도: 1 | 추론력: 1 | 한줄평: 내용의 부재로 인해 질문에 대해 아무런 답변이 이루어지지 않았다.

GraphRAG | 정확성: 2 | 완성도: 3 | 추론력: 3 | 한줄평: 관련 정보가 없지만 간접적으로 관련된 내용을 언급하며 어느 정도의 추정을 시도했다.
------------------------------------------------------------
질문: 트럼프와 이란의 관계를 설명해줘
유형: local


Q: 트럼프와 이란의 관계를 설명해줘
[Naive RAG] | 정확성: 4 | 완성도: 3 | 추론력: 2 | 한줄평: 기본적인 사실은 담고 있으나 세부 정보와 맥락이 부족하다.  
  
[Advanced RAG] | 정확성: 4 | 완성도: 4 | 추론력: 3 | 전반적인 관계의 긴장도를 잘 설명하였으나, 세부적인 맥락에서 약간의 부족함이 있다.  
  
[GraphRAG] | 정확성: 5 | 완성도: 5 | 추론력: 4 | 사실에 정확하며 맥락과 경제적 요소까지 포괄적으로 설명하여 잘 정리된 답변이다.
------------------------------------------------------------
질문: 이란 전쟁이 한국 경제에 미치는 영향은?
유형: global


Q: 이란 전쟁이 한국 경제에 미치는 영향은?
| 방식명      | 정확성 | 완성도 | 추론력 | 한줄평                                      |
|------------|------|------|------|-------------------------------------------|
| Naive RAG  | 1    | 1    | 1 